# California Housing regression with SeeDL

Use scikit-learn to fit a model, then inspect held-out performance with one SeeDL call. New APIs in this branch: `sd.evaluate`, `sd.feature_importance`, and `sd.partial_dependence`. Existing APIs reused: `sd.see`, `sd.dataset_health`, `sd.report`, `.show()` and `.save()`.

Install this checkout with `python -m pip install -e ".[sklearn]"` and install Jupyter separately. This notebook explicitly downloads California Housing on first use via scikit-learn, then uses its local cache. The library itself never downloads data or trains models.

In [ ]:
import os
from pathlib import Path

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import seedl as sd

housing = fetch_california_housing(data_home=os.environ.get("SCIKIT_LEARN_DATA"), as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    housing.data, housing.target, test_size=0.2, random_state=42
)
output = Path("artifacts/california-housing")
output.mkdir(parents=True, exist_ok=True)
print(housing.data.shape, housing.feature_names)

## Fit in your own scikit-learn code

These 1990 California census measurements are a historical benchmark, not current house-price estimates. The target is median house value in units of **100,000 USD**, with values capped in the source benchmark around 5.0. Features use their original documented units; preprocessing changes coefficient scaling.

In [ ]:
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)  # Explicit user training; SeeDL does not fit this model.
inspection = sd.see(model, backend="svg")
inspection.save(output / "pipeline.html")

## One call for test-set evaluation

MAE and RMSE below use 100,000-USD target units; MSE uses squared target units. R² is dimensionless and may be negative. Each metric uses every supplied test row, while scatter plots are bounded previews. `split="test"` records your declaration: SeeDL cannot detect data leakage or prove independence.

In [ ]:
result = sd.evaluate(model, X_test, y_test, split="test", backend="svg")
result.metadata["Target units"] = "100,000 USD (median house value)"
result.notes.append(
    "California Housing, sklearn benchmark: target units 100,000 USD; historical 1990 census, source target cap applies."
)
result.save(output / "evaluation.html")
result

In [ ]:
result.metrics
# Example: monetary RMSE in USD = result.metrics["rmse"] * 100_000

## Select a plot without rerunning prediction

Residual = actual − predicted. A zero-centered plot can still hide regional or subgroup errors; this notebook does not claim fairness or deployment readiness.

In [ ]:
residuals = result.plot("residuals")
residuals.save(output / "residuals.svg")
residuals

## Optional interpretation on explicit reference data

Permutation importance measures held-out score changes, not causality. `neg_mean_absolute_error` is a signed scikit-learn scorer: importance is the decrease in that score. Correlated features can share importance. Standard deviations describe repeated permutations, not confidence intervals.

In [ ]:
importance = sd.feature_importance(
    model,
    X_test,
    y_test,
    method="permutation",
    split="test",
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    max_samples=256,
    backend="svg",
)
importance.save(output / "permutation.html")

Partial dependence varies original **MedInc** through the entire pipeline and averages over a bounded reference sample. It may create unrealistic combinations with correlated features. PDP and ICE are model-response summaries, not causal effects.

In [ ]:
dependence = sd.partial_dependence(
    model,
    X_test,
    ["MedInc"],
    kind="both",
    max_samples=128,
    max_ice=6,
    grid_resolution=20,
    backend="svg",
)
dependence.save(output / "partial-dependence.html")

## Reuse existing SeeDL data checks and reports

`dataset_health` is an existing independent data-inspection API, not a new sklearn metric. No fitting occurs when views are combined or exported.

In [ ]:
health = sd.dataset_health(X_train.to_numpy(), backend="svg")
report = sd.report(
    title="California Housing held-out evaluation", views=[result, importance, dependence, health]
)
report.save(output / "report.html")
print("Offline files:", output.resolve())

## Learning curves require deliberate extra training

Use training data only for model selection. This cell does nothing by default; enabling it explicitly permits scikit-learn to fit **clones**. The original model stays unchanged, and the final test set is not used.

In [ ]:
RUN_EXTRA_TRAINING = False
if RUN_EXTRA_TRAINING:
    curve = sd.learning_curve(
        model,
        X_train,
        y_train,
        allow_refit=True,
        cv=3,
        train_sizes=[0.5, 1.0],
        max_samples=1000,
        backend="svg",
    )
    curve.save(output / "learning-curve.html")

Source: [California Housing dataset documentation](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset). Numerical metrics come from scikit-learn; saved HTML/SVG files work offline and embed Google Sans / JetBrains Mono. See `docs/CLASSICAL_ML.md` for supported estimators and limitations.